interesting traits -> >= 1 genetic association

In [32]:
import pandas as pd
import os
import glob
import pickle

In [ ]:
data_dir = "../data/"
files = glob.glob(f"{data_dir+"association_by_datasource_direct"}/*.parquet")

In [23]:

genetic_association_sources = [
    'genomics_england','orphanet','gwas_credible_sets','eva',
    'uniprot_literature', 'gene_burden', 'uniprot_variants',
    'gene2phenotype', 'clingen'
]

score_thresholds = {
    'genomics_england':0.5,
    'orphanet': 1,
    'gwas_credible_sets':0.5,
    'eva': 0.5,
    'uniprot_literature': 0.5, 
    'gene_burden': 0.25, 
    'uniprot_variants': 0.5,
    'gene2phenotype': 0.5, 
    'clingen': 0.5 
}

In [ ]:
chunk_size = 5
filtered_chunks = []
#assoc_chunk = pd.concat([pd.read_parquet(f) for f in files[indexes[0]:indexes[-1]+1]], ignore_index=True)


In [24]:
for i in range(0, len(files), chunk_size):
    print(f"Reading and filtering chunk: {i}")
    chunk_files = files[i:i + chunk_size]  # slicing handles the end boundary cleanly
    assoc_chunk = pd.concat([pd.read_parquet(f) for f in chunk_files], ignore_index=True)
    assoc_chunk = assoc_chunk[assoc_chunk['aggregationValue'].isin(genetic_association_sources)].reset_index(drop=True)
    assoc_chunk['threshold'] = assoc_chunk.apply(lambda x: score_thresholds[x['aggregationValue']], axis=1)
    assoc_chunk = assoc_chunk[assoc_chunk['associationScore'] >= assoc_chunk['threshold']].reset_index(drop=True)
    filtered_chunks.append(assoc_chunk.copy())  # no need to wrap in pd.DataFrame()

Reading and filtering chunk: 0
Reading and filtering chunk: 5
Reading and filtering chunk: 10
Reading and filtering chunk: 15
Reading and filtering chunk: 20
Reading and filtering chunk: 25
Reading and filtering chunk: 30
Reading and filtering chunk: 35
Reading and filtering chunk: 40
Reading and filtering chunk: 45
Reading and filtering chunk: 50


In [25]:
genetic_associations =  pd.concat(filtered_chunks)

In [28]:
genetic_associations.shape

(250578, 9)

In [31]:
traits = list(genetic_associations['diseaseId'].unique())

In [43]:
with open(os.path.join("..","aux_data","traits_geq_oneGenAssoc.pkl"),"wb") as file:
    pickle.dump(traits, file)